# SILVA Monotone Graph Equilibrium

This lab derives the monotone operator parameterization, its forward-backward
step, the normalized graph operator, and the corresponding SILVA family. It
checks an exact graph-elliptic dataset, trains a compact node field, and tests
node relabeling. The mechanism follows monotone implicit graph networks [47]
and remains inside the canonical `silva_monotone_graph_equilibrium` family.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [1](https://jseluis.github.io/silva-networks/paper/references/#ref-1), [4](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [47](https://jseluis.github.io/silva-networks/paper/references/#ref-47). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import (
    SILVAMonotoneGraphEquilibrium,
    SolverConfig,
    make_monotone_chain_dataset,
    normalized_laplacian_field,
)

torch.manual_seed(210)

## 1. Graph Operator and Shape Contract

For node state $Z\in\mathbb R^{N\times d}$, define

$$
G=\frac12\left(I-D^{-1/2}AD^{-1/2}\right),
\qquad GZ\in\mathbb R^{N\times d}.
$$

The factor $1/2$ places the normalized-Laplacian spectrum in $[0,1]$. The
package accepts a directed edge list; bidirectional edges represent an
undirected graph.

In [ ]:
data = make_monotone_chain_dataset(nodes=12, channels=1, diffusion=0.6, seed=21)
graph_field = normalized_laplacian_field(data.target, data.edge_index)
equation_error = data.equation_residual().abs().max()

assert data.source.shape == data.target.shape == (12, 1)
assert graph_field.shape == data.target.shape
assert equation_error < 1e-6
print("edges:", data.edge_index.shape[1])
print("maximum graph-equation residual:", float(equation_error))

## 2. Monotone Channel Parameterization

The channel operator is not an unconstrained matrix. It is formed as

$$
W=(1-m)I-CC^T+F-F^T,
\qquad m>0.
$$

Because the skew term vanishes in the symmetric part,

$$
I-\frac{W+W^T}{2}=mI+CC^T\succeq mI.
$$

The smallest eigenvalue is therefore a directly testable certificate. This is
the stability constraint represented by
`SILVAMonotoneGraphTransition.monotonicity_certificate()`.

## 3. Forward-Backward Step as a SILVA Transition

With source $B(X)$ and proximal activation, one operator-splitting step is

$$
Z^{k+1}=\operatorname{prox}_{\alpha f}
\left((1-\alpha)Z^k+\alpha(WGZ^k+B(X))\right).
$$

In SILVA, $B(X)$ is the source branch, $WGZ$ is the graph-local branch, and
the proximal map is the output nonlinearity. Reusing this transition until
convergence produces one implicit graph point rather than an explicit stack.

In [ ]:
model = SILVAMonotoneGraphEquilibrium(
    in_dim=1,
    state_dim=6,
    out_dim=1,
    margin=0.15,
    step_size=0.7,
    config=SolverConfig(solver="picard", max_iter=25, tol=1e-6),
)
initial = model(data.source, data.edge_index, return_result=True)

assert initial.output.shape == data.target.shape
assert initial.monotonicity_certificate >= 0.15 - 1e-6
print("certificate:", float(initial.monotonicity_certificate))
print("equilibrium residual:", initial.solver_result.residual)

## 4. Tiny Equation-Supervised Task

The target solves

$$
(I+\nu G)u=s.
$$

The loss below teaches the readout and equilibrium transition to approximate
that solution. This is a deterministic small-scale reproduction of the graph
mechanism, not a citation-network benchmark.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-2)
losses = []
for _ in range(12):
    optimizer.zero_grad()
    prediction = model(data.source, data.edge_index)
    loss = torch.nn.functional.mse_loss(prediction, data.target)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))

trained = model(data.source, data.edge_index, return_result=True)
print("initial/final task loss:", losses[0], losses[-1])
print("final equilibrium residual:", trained.solver_result.residual)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(7.2, 2.7))
axes[0].plot(losses, marker="o", markersize=2)
axes[0].set(xlabel="optimization step", ylabel="MSE", yscale="log")
axes[1].plot(data.target[:, 0], label="exact", linewidth=2)
axes[1].plot(trained.output.detach()[:, 0], "--", label="SILVA")
axes[1].set(xlabel="node", ylabel="field")
axes[1].legend()
figure.tight_layout()
plt.show()

## 5. Node Relabeling

For permutation matrix $P$, a graph equilibrium must satisfy

$$
F(PX,PEP^T)=PF(X,E).
$$

The edge list must be relabeled with the nodes. This is different from
permuting features while leaving graph topology unchanged.

In [ ]:
permutation = torch.tensor([7, 1, 10, 3, 5, 9, 0, 11, 2, 8, 4, 6])
inverse = torch.empty_like(permutation)
inverse[permutation] = torch.arange(permutation.numel())
permuted_edges = inverse[data.edge_index]

with torch.no_grad():
    reference = model(data.source, data.edge_index)
    relabeled = model(data.source[permutation], permuted_edges)
equivariance_error = (relabeled - reference[permutation]).abs().max()
assert equivariance_error < 1e-5
print("relabeling error:", float(equivariance_error))

## 6. What to Report

Record the graph normalization, directed-edge convention, margin $m$, proximal
map, forward-backward step size, solver, fixed-point residual, and task metric.
The monotonicity certificate diagnoses the parameterization; it does not replace
the numerical residual or downstream accuracy.

## From 21 Silva Monotone Graph Equilibrium to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | one latent vector per node or entity |
| Condition | node features, edges, edge attributes, and graph batches |
| Repeated computation | a source-injected graph message or monotone graph transition |
| Required invariants | node relabeling equivariance, graph boundaries, and state shape |
| Replaceable components | input projection, message field, global field, transition, pooling, and head |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


### Replace the Monotone Transition and Certificate

```python
model = SILVAMonotoneGraphEquilibrium(
    in_dim=input_dim,
    state_dim=width,
    out_dim=output_dim,
    transition=my_monotone_transition,
    certificate=my_monotonicity_certificate,
    readout=my_readout,
    config=solver_config,
)
```

The transition has signature `(state, inputs, edge_index, edge_weight)` and
must preserve node count and state width. A custom certificate remains separate
from numerical convergence diagnostics.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**node/graph error, physical graph residual, and fixed-point residual**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **node count, edge count, feature width, and number of graphs**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '21_silva_monotone_graph_equilibrium.ipynb',
    "state": 'one latent vector per node or entity',
    "condition": 'node features, edges, edge attributes, and graph batches',
    "transition": 'a source-injected graph message or monotone graph transition',
    "invariants": 'node relabeling equivariance, graph boundaries, and state shape',
    "compact_metric": 'node/graph error, physical graph residual, and fixed-point residual',
    "scale_axis": 'node count, edge count, feature width, and number of graphs',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


## Where to Go Next

| Question | Page |
| --- | --- |
| How is the monotone parameterization derived? | [Advanced Equilibrium Families](https://jseluis.github.io/silva-networks/learn/advanced-equilibrium-families/#monotone-graph-equilibrium) |
| Which exact chain equation is checked? | [Advanced Equilibrium Datasets](https://jseluis.github.io/silva-networks/learn/advanced-equilibrium-datasets/#monotone-chain) |
| Which graph classes and helpers are public? | [Advanced Equilibria API](https://jseluis.github.io/silva-networks/api/advanced_equilibria/) |
